In [185]:
import numpy as np
import pandas as pd
import os
import sys
import pandas as pd

In [186]:
crops = pd.read_csv('data/FAOSTAT_crops.csv')
df_faostat_prices = pd.read_csv("data/FAOSTAT_prices.csv")

In [187]:
def crops_format(df, name):
    df = crops[crops['Item'] == name]
    df = df[['Area', 'Unit', 'Element', 'Item', 'Year', 'Value']]
    df_ = pd.pivot_table(df, index=['Item', 'Year'], columns='Element', values='Value').reset_index()
    df_ = df_.rename(columns={'Year': 'date'})
    df_ = df_[['Item', 'date', 'Area harvested', 'Yield', 'Production']]
    return df_

grapes = crops_format(crops, "Grapes")
print(grapes.head(10))
tomatoes = crops_format(crops, "Tomatoes")
print(tomatoes.head(10))
wheat = crops_format(crops, "Wheat")
print(wheat.head(10))
maize = crops_format(crops, "Maize (corn)")
print(maize.head(10))
sunflower = crops_format(crops, "Sunflower seed")
print(sunflower.head(10))

Element    Item  date  Area harvested   Yield  Production
0        Grapes  1980       1657400.0  4055.4   6721400.0
1        Grapes  1981       1657000.0  3260.4   5402503.0
2        Grapes  1982       1658000.0  3635.7   6028000.0
3        Grapes  1983       1645330.0  3111.3   5119100.0
4        Grapes  1984       1587250.0  3548.4   5632200.0
5        Grapes  1985       1552387.0  3510.9   5450240.0
6        Grapes  1986       1531200.0  3828.7   5862500.0
7        Grapes  1987       1514000.0  4204.0   6364800.0
8        Grapes  1988       1439900.0  2611.6   3760400.0
9        Grapes  1989       1434900.0  3508.0   5033600.0
Element      Item  date  Area harvested    Yield  Production
0        Tomatoes  1980         60700.0  35375.6   2147300.0
1        Tomatoes  1981         59600.0  36223.2   2158900.0
2        Tomatoes  1982         59300.0  38059.0   2256900.0
3        Tomatoes  1983         59600.0  39406.0   2348600.0
4        Tomatoes  1984         65100.0  38567.7   251075

Unités de mesure :
- Yield (in kg/ha)
- Production (in t)
- Area harvested (in ha)



Répartition de la ferme :

- total = 70 ha
- 10% tomatoes
- 20% sunflower
- 30% maize
- 10% grapes
- 30% wheat

In [188]:
SPI = pd.read_csv('data/SPI.csv')



SPI['time'] = pd.to_datetime(SPI['time'])

SPI = SPI.rename(columns={'SPI3': 'SPI'})

SPI['year'] = SPI['time'].dt.year
SPI_index = SPI.groupby('year')['SPI'].mean().reset_index()
SPI_index = SPI_index.rename(columns={'year': 'date'})

# Formater la date avec l'année-mois-jour heure (01-01 00:00:00)
SPI_index['date'] = pd.to_datetime(SPI_index['date'].astype(str))
print(SPI_index.head(10))

        date       SPI
0 1980-01-01  0.696258
1 1981-01-01  0.583559
2 1982-01-01  0.457449
3 1983-01-01 -0.535935
4 1984-01-01  1.113628
5 1985-01-01  0.072208
6 1986-01-01 -0.610300
7 1987-01-01 -0.185411
8 1988-01-01  1.306129
9 1989-01-01  1.614868


In [189]:
df_faostat_prices = df_faostat_prices.loc[:,["Item Code (CPC)", "Item", "Year", "Unit", "Value"]]
df_faostat_prices.rename(columns={"Year": "date"}, inplace=True)

In [190]:
grapes_prices = df_faostat_prices[df_faostat_prices["Item"] == "Grapes"].drop(columns=["Item Code (CPC)"])
maize_prices = df_faostat_prices[df_faostat_prices["Item"] == "Maize (corn)"].drop(columns=["Item Code (CPC)"])
sunflower_prices = df_faostat_prices[df_faostat_prices["Item"] == "Sunflower seed"].drop(columns=["Item Code (CPC)"])
tomatoes_prices = df_faostat_prices[df_faostat_prices["Item"] == "Tomatoes"].drop(columns=["Item Code (CPC)"])
wheat_prices = df_faostat_prices[df_faostat_prices["Item"] == "Wheat"].drop(columns=["Item Code (CPC)"])


In [191]:
def merge_price(df, price_df):
    df = pd.merge(df, price_df[['date', 'Value']], on='date', how='left')
    df = df.rename(columns={'Value': 'Price'})
    df.dropna(inplace=True)
    return df

grapes = merge_price(grapes, grapes_prices)
tomatoes = merge_price(tomatoes, tomatoes_prices)
wheat = merge_price(wheat, wheat_prices)
maize = merge_price(maize, maize_prices)
sunflower = merge_price(sunflower, sunflower_prices)

In [192]:
ipampa_overall_index = pd.read_csv("data/IPAMPA_overall_index.csv", sep=";")
ipampa_overall_index.rename(columns={"Label": "Year", "Annual agricultural means of production purchasing price index (IPAMPA) - Overall index": "IPAMPA_overall_index"}, inplace=True)
print(ipampa_overall_index.head(10))

   Year  IPAMPA_overall_index
0  2025                 124.7
1  2024                 125.4
2  2023                 131.2
3  2022                 133.7
4  2021                 109.2
5  2020                 100.0
6  2019                 101.7
7  2018                 100.2
8  2017                  96.7
9  2016                  95.5


### Calul des revenus (outputs)


In [193]:
weights = {"Grapes": 0.2, "Maize (corn)": 0.2, "Sunflower seed": 0.2, "Tomatoes": 0.2, "Wheat": 0.2}
total = 70 #ha
ha_grapes = 70* weights["Grapes"]
ha_maize = 70* weights["Maize (corn)"]
ha_sunflower = 70* weights["Sunflower seed"]
ha_tomatoes = 70* weights["Tomatoes"]
ha_wheat = 70* weights["Wheat"]

In [194]:
grapes["Yield_tot"] = grapes["Yield"] * ha_grapes * 0.001 #en tonnes
maize["Yield_tot"] = maize["Yield"] * ha_maize * 0.001 #en tonnes
sunflower["Yield_tot"] = sunflower["Yield"] * ha_sunflower * 0.001 #en tonnes
tomatoes["Yield_tot"] = tomatoes["Yield"] * ha_tomatoes * 0.001 #en tonnes
wheat["Yield_tot"] = wheat["Yield"] * ha_wheat * 0.001 #en tonnes
print(grapes.head(10))

      Item  date  Area harvested   Yield  Production  Price  Yield_tot
11  Grapes  1991       1378892.0  3769.0   5197000.0  528.4    52.7660
12  Grapes  1992       1309169.0  4397.7   5757300.0  532.6    61.5678
13  Grapes  1993       1235000.0  3698.5   4567600.0  411.1    51.7790
14  Grapes  1994       1192600.0  2728.8   3254360.0  564.8    38.2032
15  Grapes  1995       1160100.0  2887.8   3350100.0  735.4    40.4292
16  Grapes  1996       1122400.0  4431.2   4973600.0  495.0    62.0368
17  Grapes  1997       1126989.0  4901.0   5523400.0  651.1    68.6140
18  Grapes  1998       1117900.0  4604.0   5146810.0  542.3    64.4560
19  Grapes  1999       1115282.0  5028.0   5607660.0  485.4    70.3920
20  Grapes  2000       1167703.0  5600.6   6539812.0  402.7    78.4084


In [195]:
grapes["Revenue"] = grapes["Yield_tot"] * grapes["Price"]
wheat["Revenue"] = wheat["Yield_tot"] * wheat["Price"]
maize["Revenue"] = maize["Yield_tot"] * maize["Price"]
sunflower["Revenue"] = sunflower["Yield_tot"] * sunflower["Price"]
tomatoes["Revenue"] = tomatoes["Yield_tot"] * tomatoes["Price"]


In [196]:
Revenue = pd.concat([grapes[['date', 'Revenue']], wheat[['date', 'Revenue']], maize[['date', 'Revenue']], sunflower[['date', 'Revenue']], tomatoes[['date', 'Revenue']]])
Revenue = Revenue.groupby('date')['Revenue'].sum().reset_index()
print(Revenue.head(10))


   date       Revenue
0  1991  330272.99130
1  1992  313953.73352
2  1993  310061.85938
3  1994  312857.94862
4  1995  327785.36672
5  1996  385792.75056
6  1997  486224.67264
7  1998  515418.27498
8  1999  488397.86744
9  2000  556252.23696


### Calcul des coûts (inputs)

Nous allons utiliser un indice de prix de coûts calculé par l'INSEE : IPAMPA
Cet indice est en base 100 = 2020

Prix en 2020:
- Fertilisation (Azote, P, K) = 175 € (35 %)
- Phytosanitaires = 45 € (29 %)
- Semences = 70 € (14 %)
- Carburant (GNR) = 65 € (13 %)
- Séchage / Divers = 45 € (9 %)

Total = 500 euros de coûts en 2020
